# Data preparation for Llama
Stage 2 & 3

In [ ]:
%pip install seaborn

## A Overview

In [ ]:
from pathlib import Path
import pandas as pd

# plotting the distribution of confidence scores by class for failed predictions
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

import configuration

from src import hf_utils, setup
from src.models import bert

In [ ]:
datasets_path = Path("..") / "data"/"splitted" / "stage_2"
strategy = "crossed" # "crossed" or "isolated"
times = 8 # how many times to sample the false positives to balance the dataset
df_predictions = pd.read_csv(datasets_path / strategy / "llama_pre_bert_sets_with_predictions.csv")
df_predictions.shape

In [ ]:
df_failed_predictions = df_predictions[df_predictions["informative"] != df_predictions["predicted"]]
print(df_failed_predictions.shape)
df_failed_predictions.head()

In [ ]:

sns.histplot(data=df_failed_predictions, x="confidence", hue="informative", bins=20, kde=True)
plt.title("Distribution of Confidence Scores for Failed Predictions")
plt.xlabel("Confidence Score")
plt.ylabel("Count")
plt.legend(title="Informative")
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
# common_norm=False ensures each class's density is scaled independently
sns.kdeplot(
    data=df_failed_predictions, 
    x="confidence", 
    hue="informative", 
    fill=True, 
    common_norm=False, 
    alpha=0.5
)
plt.title("Density Distribution of Confidence Scores by Informative Class")
plt.xlabel("Confidence Score")
plt.ylabel("Density")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=df_failed_predictions, 
    x="informative", 
    y="confidence",
    palette="Set2"
)
plt.title("Boxplot of Confidence Scores by Informative Class")
plt.xlabel("Is Informative?")
plt.ylabel("Confidence Score")
plt.show()

## B. Re-distribute for cross strategy

In [ ]:
df_false_negatives = df_failed_predictions[df_failed_predictions["informative"]]
df_false_positives = df_failed_predictions[~df_failed_predictions["informative"]]

n = len(df_false_negatives) * times
df_false_positives = df_false_positives.sample(n=n, random_state=setup.RANDOM_SEED)

df = pd.concat([df_false_negatives, df_false_positives])
df.shape


In [ ]:
df.shape

In [ ]:
# plot sampled_df to check the distribution of confidence scores
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x="confidence", bins=20, hue="informative", kde=True)
plt.title("Distribution of Confidence Scores for Sampled Failed Predictions")
plt.xlabel("Confidence Score")
plt.ylabel("Count")
plt.legend(title="Informative")
plt.show()

## C. Spliting

In [ ]:
df_train, df_test = train_test_split(df, test_size=0.2, random_state=setup.RANDOM_SEED, stratify=df["informative"])

df_test, df_validation = train_test_split(df_test, test_size=0.5, random_state=setup.RANDOM_SEED, stratify=df_test["informative"])
print(f"Train set shape: {df_train.shape}")
print(f"Validation set shape: {df_validation.shape}")
print(f"Test set shape: {df_test.shape}")

In [ ]:
save_path = datasets_path / strategy / f"{times}_times"
save_path.mkdir(parents=True, exist_ok=True)
df_train.to_csv(save_path / "df_train.csv", index=False)
df_validation.to_csv(save_path / "df_validation.csv", index=False)
df_test.to_csv(save_path / "df_test.csv", index=False)